# Feature Extraction Validation

Validates the output of `vm-extract-air` (and optionally `vm-extract-struct`).

Checks:
  1. Shape and column inventory
  2. Missing values
  3. Near-constant features
  4. Depth-correlation (Spearman) for every feature
  5. Per-feature distributions grouped by depth level
  6. CWT and DWT energy sanity (should increase with depth for some bands)
  7. Recording-level consistency (within-run variance vs between-run variance)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

from vm_micro.utils.paths import PROJECT_ROOT

plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

# ── CONFIGURE THESE ──────────────────────────────────────────────────────────
FEATURES_CSV = Path(PROJECT_ROOT / "data/features/structure/features_structure_v2.csv")
MODALITY = "structure"  # or "structure"
TOP_N_CORR = 30  # how many top-correlated features to plot
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(FEATURES_CSV)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Depth levels present: {sorted(df['depth_mm'].unique())}")
print(f"Runs present: {df['recording_root'].nunique()} unique runs")

## 1 · Shape and column inventory

In [ ]:
META_COLS = {
    "modality",
    "record_name",
    "recording_root",
    "depth_mm",
    "step_idx",
    "duration_s",
    "sr_hz",
    "sr_hz_native",
    "sr_hz_used",
    "ds_rate",
    "file_path",
}
feat_cols = [c for c in df.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(df[c])]
print(f"\nFeature columns: {len(feat_cols)}")

# Family breakdown
families = {}
for c in feat_cols:
    prefix = c.split("_")[0]
    families[prefix] = families.get(prefix, 0) + 1
print("\nFeatures per family:")
for k, v in sorted(families.items(), key=lambda x: -x[1]):
    print(f"  {k:20s}: {v}")

## 2 · Missing values

In [ ]:
miss = df[feat_cols].isna().mean().sort_values(ascending=False)
n_missing = (miss > 0).sum()
print(f"\nFeatures with any missing values: {n_missing}")
if n_missing > 0:
    print(miss[miss > 0].to_string())

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(miss.values, bins=50)
ax.set_xlabel("Missing fraction")
ax.set_ylabel("Number of features")
ax.set_title("Missing value distribution across features")
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_missing.png")
plt.show()

## 3 · Near-constant features

In [ ]:
std_vals = df[feat_cols].std()
near_zero = std_vals[std_vals < 1e-8]
print(f"\nNear-constant features (std < 1e-8): {len(near_zero)}")
if len(near_zero):
    print(near_zero.to_string())

## 4 · Spearman correlation with depth

In [ ]:
corr = pd.Series(
    {c: abs(float(spearmanr(df[c].fillna(0), df["depth_mm"]).statistic)) for c in feat_cols},
    name="|spearman_r|",
).sort_values(ascending=False)

corr.to_csv(FEATURES_CSV.parent / f"{MODALITY}_features_corr.csv")

print("\nTop 10 features by |Spearman r| with depth_mm:")
print(corr.head(10).to_string())
print(f"\nFeatures with |r| > 0.5:  {(corr > 0.5).sum()}")
print(f"Features with |r| > 0.3:  {(corr > 0.3).sum()}")
print(f"Features with |r| < 0.05: {(corr < 0.05).sum()}  (likely noise)")

fig, ax = plt.subplots(figsize=(10, 4))
corr.values[:TOP_N_CORR][::-1]
ax.barh(range(TOP_N_CORR), corr.values[:TOP_N_CORR][::-1])
ax.set_yticks(range(TOP_N_CORR))
ax.set_yticklabels(corr.index[:TOP_N_CORR][::-1], fontsize=7)
ax.set_xlabel("|Spearman r| with depth_mm")
ax.set_title(f"Top {TOP_N_CORR} features by depth correlation")
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_spearman_top.png")
plt.show()

# Histogram of all correlations
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(corr.values, bins=50)
ax.axvline(0.1, color="orange", linestyle="--", label="|r|=0.1")
ax.axvline(0.3, color="green", linestyle="--", label="|r|=0.3")
ax.set_xlabel("|Spearman r| with depth_mm")
ax.set_ylabel("Number of features")
ax.set_title("Distribution of depth correlations — all features")
ax.legend()
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_spearman_dist.png")
plt.show()

## 4b · Duration-partialling test

Checks whether the apparent feature-vs-depth association survives after controlling for `duration_s`.

For every feature, this section computes:
- raw Spearman correlation with depth
- Spearman correlation with duration
- partial Spearman correlation with depth after partialling out duration

Interpretation boundary used here:
- `|partial_r| > 0.15` = minimally useful after duration control
- `|partial_r| > 0.20` = stronger evidence that the feature carries depth information beyond duration


In [ ]:
from scipy.stats import pearsonr, rankdata

assert "duration_s" in df.columns, (
    "The notebook expects a duration_s column for duration partialling."
)


def partial_spearman_duration(x, y, z):
    """Partial Spearman via rank residualization.

    Returns
    -------
    r_partial : float
        Partial Spearman correlation between x and y given z.
    p_value : float
        Pearson p-value on the rank residuals.
    n_used : int
        Number of non-missing rows used.
    """
    tmp = pd.DataFrame({"x": x, "y": y, "z": z}).replace([np.inf, -np.inf], np.nan).dropna()
    n_used = len(tmp)
    if n_used < 4:
        return np.nan, np.nan, n_used

    if tmp["x"].nunique() < 2 or tmp["y"].nunique() < 2 or tmp["z"].nunique() < 2:
        return np.nan, np.nan, n_used

    xr = rankdata(tmp["x"].to_numpy()).astype(float)
    yr = rankdata(tmp["y"].to_numpy()).astype(float)
    zr = rankdata(tmp["z"].to_numpy()).astype(float)

    Z = np.column_stack([np.ones(n_used), zr])
    beta_x = np.linalg.lstsq(Z, xr, rcond=None)[0]
    beta_y = np.linalg.lstsq(Z, yr, rcond=None)[0]

    x_res = xr - Z @ beta_x
    y_res = yr - Z @ beta_y

    r_partial, p_value = pearsonr(x_res, y_res)
    return float(r_partial), float(p_value), n_used


duration_depth_r = float(spearmanr(df["duration_s"], df["depth_mm"], nan_policy="omit").statistic)
print(f"Duration vs depth Spearman r: {duration_depth_r:.3f}")

rows = []
for feat in feat_cols:
    raw_r = float(spearmanr(df[feat], df["depth_mm"], nan_policy="omit").statistic)
    dur_r = float(spearmanr(df[feat], df["duration_s"], nan_policy="omit").statistic)
    part_r, part_p, n_used = partial_spearman_duration(df[feat], df["depth_mm"], df["duration_s"])

    rows.append(
        {
            "feature": feat,
            "raw_r_depth": raw_r,
            "abs_raw_r": abs(raw_r),
            "r_duration": dur_r,
            "abs_r_duration": abs(dur_r),
            "partial_r_depth_given_duration": part_r,
            "abs_partial_r": abs(part_r) if pd.notna(part_r) else np.nan,
            "partial_p_value": part_p,
            "n_used": n_used,
        }
    )

duration_partial_df = pd.DataFrame(rows)
duration_partial_df["delta_abs_r"] = (
    duration_partial_df["abs_raw_r"] - duration_partial_df["abs_partial_r"]
)
duration_partial_df["retained_ratio"] = duration_partial_df["abs_partial_r"] / duration_partial_df[
    "abs_raw_r"
].replace(0, np.nan)
duration_partial_df["pass_partial_015"] = duration_partial_df["abs_partial_r"] > 0.15
duration_partial_df["pass_partial_020"] = duration_partial_df["abs_partial_r"] > 0.20
duration_partial_df["duration_link_gt_030"] = duration_partial_df["abs_r_duration"] > 0.30
duration_partial_df = duration_partial_df.sort_values(
    ["abs_partial_r", "abs_raw_r"], ascending=False
).reset_index(drop=True)

duration_partial_df.to_csv(
    FEATURES_CSV.parent / f"{MODALITY}_duration_partialling.csv", index=False
)

print("\nTop 10 features by |partial r| with depth after controlling for duration:")
print(
    duration_partial_df[
        ["feature", "abs_raw_r", "abs_r_duration", "abs_partial_r", "delta_abs_r", "retained_ratio"]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nDuration-partialling summary:")
print(
    f"Features with raw |r| > 0.20:                {(duration_partial_df['abs_raw_r'] > 0.20).sum()}"
)
print(
    f"Features with partial |r| > 0.15:            {(duration_partial_df['abs_partial_r'] > 0.15).sum()}"
)
print(
    f"Features with partial |r| > 0.20:            {(duration_partial_df['abs_partial_r'] > 0.20).sum()}"
)
print(
    f"Features keeping >= 80% of raw |r|:          {(duration_partial_df['retained_ratio'] >= 0.80).sum()}"
)
print(
    f"Features strongly linked to duration (>|0.30|): {(duration_partial_df['abs_r_duration'] > 0.30).sum()}"
)
print(
    f"Median drop in |r| after partialling:        {duration_partial_df['delta_abs_r'].median():.3f}"
)
print(
    f"Mean drop in |r| after partialling:          {duration_partial_df['delta_abs_r'].mean():.3f}"
)

collapsed = duration_partial_df[
    (duration_partial_df["abs_raw_r"] > 0.20) & (duration_partial_df["abs_partial_r"] <= 0.10)
]
if len(collapsed):
    print("\nFeatures that looked useful raw but mostly collapsed after duration control:")
    print(
        collapsed[["feature", "abs_raw_r", "abs_r_duration", "abs_partial_r", "delta_abs_r"]]
        .head(12)
        .to_string(index=False)
    )

# 1) Duration vs depth diagnostic
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="depth_mm", y="duration_s", ax=ax)
ax.set_title(f"Duration by depth level (Spearman r = {duration_depth_r:.3f})")
ax.set_xlabel("depth_mm")
ax.set_ylabel("duration_s")
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_duration_by_depth.png")
plt.show()

# 2) Raw vs partial |r| scatter
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(duration_partial_df["abs_raw_r"], duration_partial_df["abs_partial_r"], alpha=0.55, s=18)
mx = float(
    np.nanmax([duration_partial_df["abs_raw_r"].max(), duration_partial_df["abs_partial_r"].max()])
)
ax.plot([0, mx], [0, mx], linestyle="--", linewidth=1)
ax.axhline(0.15, linestyle="--", linewidth=1, label="partial |r| = 0.15")
ax.axhline(0.20, linestyle=":", linewidth=1, label="partial |r| = 0.20")
ax.set_xlabel("Raw |Spearman r| with depth")
ax.set_ylabel("Partial |r| with depth | duration")
ax.set_title("Raw vs duration-partialled depth signal")
ax.legend()
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_duration_partial_scatter.png")
plt.show()

# 3) Histogram of how much |r| dropped after partialling
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(duration_partial_df["delta_abs_r"].dropna(), bins=50)
ax.axvline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Drop in |r| after duration partialling")
ax.set_ylabel("Number of features")
ax.set_title("How much apparent depth signal was removed by controlling duration")
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_duration_partial_drop_hist.png")
plt.show()

# 4) Features tied more strongly to duration tend to lose more depth signal
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    duration_partial_df["abs_r_duration"], duration_partial_df["delta_abs_r"], alpha=0.55, s=18
)
ax.axvline(0.30, linestyle="--", linewidth=1, label="|r(feature, duration)| = 0.30")
ax.set_xlabel("|Spearman r| with duration")
ax.set_ylabel("Drop in |r| after partialling")
ax.set_title("Duration coupling vs loss of depth correlation")
ax.legend()
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_duration_coupling_vs_drop.png")
plt.show()

# 5) Dumbbell chart for strongest raw features
focus = duration_partial_df.sort_values("abs_raw_r", ascending=False).head(15).copy()
focus = focus.sort_values("abs_raw_r", ascending=True)
fig, ax = plt.subplots(figsize=(9, 6))
ypos = np.arange(len(focus))
for y, raw_v, part_v in zip(ypos, focus["abs_raw_r"], focus["abs_partial_r"]):
    ax.plot([part_v, raw_v], [y, y], linewidth=2)
ax.scatter(focus["abs_partial_r"], ypos, s=28, label="partial |r|")
ax.scatter(focus["abs_raw_r"], ypos, s=28, label="raw |r|")
ax.axvline(0.15, linestyle="--", linewidth=1)
ax.axvline(0.20, linestyle=":", linewidth=1)
ax.set_yticks(ypos)
ax.set_yticklabels(focus["feature"], fontsize=8)
ax.set_xlabel("|r|")
ax.set_title("Top raw features: before vs after duration partialling")
ax.legend()
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_duration_partial_dumbbell_top15.png")
plt.show()

# 6) Top retained features after duration control
fig, ax = plt.subplots(figsize=(9, 5))
retained_top = duration_partial_df.head(15).iloc[::-1]
ax.barh(retained_top["feature"], retained_top["abs_partial_r"])
ax.axvline(0.15, linestyle="--", linewidth=1, label="0.15")
ax.axvline(0.20, linestyle=":", linewidth=1, label="0.20")
ax.set_xlabel("|partial r| with depth | duration")
ax.set_title("Top 15 features surviving duration control")
ax.legend()
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_duration_partial_top15.png")
plt.show()

## 5 · Per-depth distributions for top features

In [ ]:
top_feats = corr.index[:12].tolist()
n_cols = 4
n_rows = (len(top_feats) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    ax = axes[i]
    for depth, grp in df.groupby("depth_mm"):
        ax.hist(grp[feat].dropna(), bins=20, alpha=0.5, label=f"{depth:.1f}")
    ax.set_title(feat, fontsize=8)
    ax.set_xlabel("")
    ax.tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Top-correlated features: distribution by depth level", y=1.01)
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_top_feat_distributions.png", bbox_inches="tight")
plt.show()

## 6 · Depth-vs-feature scatter for top 6

In [ ]:
top6 = corr.index[:6].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, feat in zip(axes.flatten(), top6):
    ax.scatter(df["depth_mm"], df[feat], alpha=0.3, s=8)
    ax.set_xlabel("depth_mm")
    ax.set_ylabel(feat, fontsize=8)
    r = float(spearmanr(df[feat].fillna(0), df["depth_mm"]).statistic)
    ax.set_title(f"{feat}\n|r|={abs(r):.3f}", fontsize=9)

plt.suptitle("Depth vs feature value (top 6 by Spearman |r|)")
plt.tight_layout()
plt.savefig(FEATURES_CSV.parent / "validation_depth_scatter.png")
plt.show()

## 7 · Recording-level consistency
Within-run variance should be lower than between-run variance for good features.

In [ ]:
top5 = corr.index[:5].tolist()
within_var = df.groupby("recording_root")[top5].var().mean()
between_var = df.groupby("recording_root")[top5].mean().var()

ratio = between_var / (within_var + 1e-18)
print("\nBetween-run variance / within-run variance (higher = more consistent):")
print(ratio.sort_values(ascending=False).to_string())

## 8 · Summary

In [ ]:
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"Total features:                     {len(feat_cols)}")
print(f"Features with |r| > 0.3:           {(corr > 0.3).sum()}")
print(f"Features with |r| > 0.1:           {(corr > 0.1).sum()}")
print(f"Features with partial |r| > 0.15:  {(duration_partial_df['abs_partial_r'] > 0.15).sum()}")
print(f"Features with partial |r| > 0.20:  {(duration_partial_df['abs_partial_r'] > 0.20).sum()}")
print(
    f"Top-30 raw features surviving >0.15 after partialling: {(duration_partial_df.sort_values('abs_raw_r', ascending=False).head(30)['abs_partial_r'] > 0.15).sum()}"
)
print(f"Near-constant features:             {len(near_zero)}")
print(f"Features with > 20% missing values: {(miss > 0.2).sum()}")
print(f"Duration vs depth Spearman r:       {duration_depth_r:.3f}")
print(f"Median |r| drop after partialling:  {duration_partial_df['delta_abs_r'].median():.3f}")

print(f"\nTop raw feature:      {corr.index[0]}  |r|={corr.iloc[0]:.3f}")
print(
    f"Top partial feature:  {duration_partial_df.loc[0, 'feature']}  |partial r|={duration_partial_df.loc[0, 'abs_partial_r']:.3f}"
)
print(
    f"2nd partial feature:  {duration_partial_df.loc[1, 'feature']}  |partial r|={duration_partial_df.loc[1, 'abs_partial_r']:.3f}"
)
print(
    f"3rd partial feature:  {duration_partial_df.loc[2, 'feature']}  |partial r|={duration_partial_df.loc[2, 'abs_partial_r']:.3f}"
)